# 00 — Overview: the whole system in one run

**Autobus** is an autonomous bus that decides **which stops to visit, and in
what order**, from passenger requests as they arrive — rather than driving a
fixed timetable.

This notebook runs the complete system once and shows what it produced. It is
the shortest path to seeing that the thing works; everything else in this
folder takes one piece of it apart.

## The notebooks in this folder

| Notebook | Answers |
|---|---|
| **00 — Overview** | What is this, and does it run? |
| 01 — The map | Where a world comes from, and how a road map becomes a routing graph |
| 02 — Demand and KPIs | Who asks for a ride, what happens to them, and how we score it |
| 03 — Policy shootout | All nine routing strategies on identical demand |
| 04 — Tuning | The knobs that matter, and two ideas that failed |
| 05 — Synthetic campaign | The headline result, and whether it is statistically sound |
| 06 — Scope condition | Where demand-responsive routing stops paying |
| 07 — CARLA driving | How the bus physically drives a real town |
| 08 — CARLA campaign | The same headline result, on real geometry |
| 09 — The harness | Reproducibility, the results store, parallelism, latency |

## How to run these

```bash
./jupyter-run.sh          # starts JupyterLab in the right conda environment
```

Notebooks 07 and 08 need a **running CARLA server**; every other notebook
needs nothing but the conda environment and runs on any machine.

### Two sizes of everything

Each notebook has a `SCALE` switch near the top:

- **`"quick"`** — the default. Minutes, not hours. Same conclusions, wider
  error bars.
- **`"full"`** — the sample sizes that produced the figures quoted in the
  submitted documents.

The outputs saved in these files were produced at `"full"`. Re-running gives
`"quick"` unless you change it. Results are cached, so raising the scale later
only runs what is missing.

### Results are not shipped

`results/runs/` is empty in the submitted copy. Every number below is
regenerated by running the notebook.

In [ ]:
# --- Bootstrap ------------------------------------------------------------
# This notebook imports the project as `src.…`, which requires the repository
# root on the import path. `./jupyter-run.sh` puts it there; opening the file
# any other way does not. This cell finds the root by walking up from the
# working directory, so the notebook runs wherever it is launched from and
# whatever folder it sits in.
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Seeded runs only reproduce when Python's string hashing is pinned: several
# policies break ties with min() over a set, whose iteration order is
# randomised per process otherwise. The simulation refuses to run unpinned;
# this says so now rather than after a long cell.
if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print("WARNING: PYTHONHASHSEED is not pinned, so seeded runs will be "
          "refused.\n         Launch with ./jupyter-run.sh, or set "
          "PYTHONHASHSEED=0.")

print(f"repository root: {ROOT}")

## What a run is made of

Five pieces, and the whole project is the way they fit together:

| Piece | Responsibility |
|---|---|
| **Map** | Turns a road network into a graph the bus can be routed over |
| **Demand generator** | Creates passenger requests over time |
| **Request pool** | The one shared store of who wants to go where, and their state |
| **Dispatch policy** | Given the bus's state, returns the next stops to visit |
| **Backend** | Moves the bus and reports arrivals — a graph, or a CARLA vehicle |

`run_trial` assembles all five and runs the simulation to completion. The
policy is the only part these notebooks vary.

In [ ]:
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.experiment import run_trial
from src.map.graph import PlanarStreetGraph
from src.map.PlanarMapProvider import PlanarMapProvider

# The map. `stop_density=0.5` on seed 42 is the 21-stop "campus map" that
# every result in this project was measured on — see notebook 01 for why the
# generator's own default is not used.
MAP = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0), seed=42,
           stop_density=0.5)

graph = PlanarStreetGraph.generate(**MAP)

# A MapProvider is the abstraction over "where the world comes from"; the
# same interface is implemented by CARLA. Notebook 01 is about this layer.
provider = PlanarMapProvider(graph)
print(f"{len(provider.get_road_nodes())} junctions, "
      f"{len(provider.get_road_edges())} road segments, "
      f"{len(provider.get_stops())} bus stops")

## One run

A small one: 30 requests, one every minute, a 40-seat bus, and passengers who
give up after 900 seconds of waiting.

`ReorderingInsertionPolicy` is the recommended policy — it keeps a schedule of
promised pickups and dropoffs, inserts each new request at its cheapest
feasible position, and then repairs the order it already has. Notebooks 03 and
04 are where that gets compared against the alternatives; here it is just
something to watch.

In [ ]:
result = run_trial(
    policy_factory=lambda g: ReorderingInsertionPolicy(g, max_wait_s=900.0),
    graph=graph,
    seed=0,
    max_requests=30,          # small, so the animation below stays watchable
    demand_interval_s=60.0,
    capacity=40,
    max_wait_s=900.0,         # how long a passenger waits before giving up
    record_animation=True,    # capture a frame per stop arrival
    keep_tick_history=True,   # keep the per-tick occupancy trace
    silent=True,
)

kpi = result.kpi
print(f"requests fulfilled   {kpi.fulfilled_count}")
print(f"requests cancelled   {kpi.cancelled_count}")
print(f"served fraction      {kpi.served_fraction:.1%}")
print(f"mean wait            {kpi.mean_wait_time:.0f} s")
print(f"mean ride            {kpi.mean_ride_time:.0f} s")
print(f"throughput           {kpi.throughput:.2f} fulfilled/min")
print(f"\nsimulated ticks      {result.tick_count}")

## Watching it

One frame per stop arrival. Stops grow and darken as passengers accumulate;
an orange ring marks a stop where somebody onboard wants to get off; the black
marker is the bus.

The interesting thing to watch for is that the bus **changes its mind** — it
re-plans every time it reaches a stop, so the order it visits stops in is not
decided in advance.

In [ ]:
from IPython.display import HTML

from src.experiment import animate_trial

# Rendered to self-contained HTML so the animation survives in the saved
# notebook. `show_animation(result)` instead gives interactive playback
# controls when you are running the notebook yourself.
HTML(animate_trial(result.animation_frames, result.transit_graph,
                   interval_ms=350, figsize=(8.0, 6.5)).to_jshtml())

## The bus filling and emptying

`occupancy` is how many passengers are aboard; `pending` is how many are
waiting at a stop. Pending climbing without falling would mean the bus is
being outrun by demand — see notebook 06, where that is the whole subject.

In [ ]:
import matplotlib.pyplot as plt

times = [s.sim_time / 60 for s in result.tick_history]
figure, axis = plt.subplots(figsize=(9, 3.2))
axis.plot(times, [s.occupancy for s in result.tick_history],
          lw=1.8, label="onboard")
axis.plot(times, [s.pending_count for s in result.tick_history],
          lw=1.8, label="waiting")
axis.set_xlabel("simulated minutes")
axis.set_ylabel("passengers")
axis.set_title("One run, from the passengers' point of view")
axis.legend(frameon=False)
axis.grid(alpha=0.3)
figure.tight_layout()

## What this did *not* show

Everything that makes the result trustworthy rather than merely plausible:

- One run is one seed. Any two policies can swap places on one seed —
  notebook 05 is about how many are needed before a difference is real.
- The bus above drove a **graph**, not a road. Notebook 07 puts the same
  policy in a CARLA vehicle in a real town.
- The KPIs printed above are averages over *fulfilled* requests, which is a
  trap worth understanding before quoting any of them. Notebook 02 explains it.

**Start with 02 if you want to understand the measurements, 03 if you want to
see the routing strategies compared, or 05 for the headline result.**